In [ ]:
from thbsplines.hierarchical_space import HierarchicalSpace
import numpy as np
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista

from dolfinx import default_real_type, default_scalar_type
rtype = default_real_type
dtype = default_scalar_type
import ufl


import numpy.typing as npt
from thbsplines.refinement import refine
from thbsplines.fenicsx.mesh import build_mesh
from thbsplines.fenicsx.functionspace import build_dofmap, create_spline_space
from thbsplines.fenicsx.solvers import solve_problem
from thbsplines.fenicsx.adaptivity import dorfler_marking
from thbsplines.fenicsx.kernels import make_linear_kernel, make_bilinear_kernel
from thbsplines.fenicsx.postprocessing import map_spline_to_legendre

In [ ]:
p0 = 2
m=4
n_refinements = 1
knots1 = np.array([-1., 0., 1.], dtype=np.float64)
knots1 = refine(knots1, p=p0, n_times=n_refinements)
# log_initial_mesh_size = np.log2(np.max(np.diff(knots1)))
err_cells = {}
hs = HierarchicalSpace(knots=[knots1, knots1, knots1], degrees=[p0])

In [ ]:
for level, cells in err_cells.items():
    hs.refine(cells, level, refine_neighbours=True, refine_T_neighbours=True, m=m)
disconnected_mesh, thb_operators, N_max, _ = build_mesh(hs=hs, dim3=True)

In [ ]:
legendre_elt = basix.ufl.element(
    "DG",
    "hexahedron",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre,
    dtype=np.float64
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata={"quadrature_degree": 8})
u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
my_x = ufl.SpatialCoordinate(disconnected_mesh)
#f = dolfinx.fem.Function(V)
#f.interpolate(lambda x: (np.tanh(9*x[1]-9*x[0]+9*x[2])+1)/9. + 1./(1.5*np.exp((10.*x[0]-6.)**2 + (10.*x[1]+7)**2 + (10.*x[2]-0.1)**2))) 
#f.interpolate(lambda x: x[0]**3+1+0.2*x[1]-0.87*x[2]*x[1] + x[1]*x[0]-0.05*x[0]**2*x[2]**2 + 10.*x[0]*x[1]**2*x[2])
#f = my_x[0]*my_x[1] - my_x[1]*my_x[2] + 0.3*my_x[2]**2
#f = (ufl.tanh(9*my_x[1]-9*my_x[0]+9*my_x[2])+1)/9. + 1./(1.5*ufl.exp(ufl.sqrt((10.*my_x[0]-6.)**2 + (10.*my_x[1]+7.)**2 + (10.*my_x[2]-0.1)**2))) 
f = 1./(1.5*ufl.exp(ufl.sqrt((10.*my_x[0]-6.)**2 + (10.*my_x[1]+7.)**2 + (10.*my_x[2]-0.1)**2))) 
#f = 1./(1.5*ufl.exp((10.*my_x[0]-6.)**2 + (10.*my_x[1]+7.)**2 + (10.*my_x[2]-0.1)**2))
#f = my_x[2]*my_x[1]**3+my_x[0]*(my_x[1]-0.7)**2 + 0.3*my_x[2]-0.1
a0 = ufl.inner(u, v) * dx_custom
f0 = ufl.inner(f, v)*dx_custom
msh = disconnected_mesh
f_square_integral = dolfinx.fem.assemble_scalar(dolfinx.fem.form(ufl.inner(f,f)*dx_custom, dtype=np.float64))
f_sq_integral = np.sqrt(disconnected_mesh.comm.allreduce(f_square_integral, op=MPI.SUM))

dofmap, padded_cells_to_dofs = build_dofmap(hierarchical_space=hs, mesh=disconnected_mesh, N_max=N_max, morton=True)

In [ ]:
M = hs._bezier_to_legendre(degree = p0)
S_indices = np.arange(p0+1, dtype=np.float64)
# scaling for unnormalised Legendre basis polynomials
S_inv = (1./np.sqrt(2.*S_indices+1.))*np.identity(p0+1, dtype=np.float64) # Scaling factor, since fenicsx uses orthonormal legendre polynomials
T = np.asfortranarray(np.kron(np.kron(M, M), M).T @ np.kron(np.kron(S_inv, S_inv), S_inv), dtype=np.float64)
local_dofs_size = T.shape[1]

operator_shape = (N_max, local_dofs_size)
# Create a custom space that holds the content of each matrix for the relevant cell.
# degree 0 because the value is constant over each cell
C_element = basix.ufl.element(
    "DG",
    cell='hexahedron',
    degree=0,
    shape = operator_shape,
    dtype=np.float64
)
C_space = dolfinx.fem.functionspace(disconnected_mesh, C_element)# ("DG", 0, operator_shape, np.float32))
C_func = dolfinx.fem.Function(C_space, dtype=np.float64)

num_cells_local = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_local
indices = np.arange(num_cells_local, dtype=np.int32)
# To make sure that each matrix is assigned to the correct cell
midpoints: npt.NDArray[np.float_] = dolfinx.mesh.compute_midpoints(disconnected_mesh, disconnected_mesh.topology.dim, indices)
c_values = C_func.x.array.reshape((-1, N_max, T.shape[1]))
for local_idx, midpoint in enumerate(midpoints):
    #print(f"midpoint = {midpoint}")
    level, idx = hs.hmesh.find_active_cell(midpoint[:hs.dim])
    #print(f"midpoint = {midpoint}, level={level}, idx={idx}")
    mat: npt.NDArray = thb_operators[level, idx] @ hs.level_spaces[level].get_bezier_operator(idx).astype(np.float64)
    #mat = hs.local_multi_level_extraction_operator(idx, level, level) @ hs.level_spaces[level].get_bezier_operator(idx)
    # print(mat.shape)
    real_k, n_cols = mat.shape

    if real_k<N_max:
        padding_size = N_max - real_k
        
        #mat_padded = np.vstack((mat, np.zeros((padding_size, mat.shape[1])) ))
        Ci = mat#_padded
    else:
        padding_size=0
        Ci = mat
    
    # is a view of C_func.x.array, therefore we modify the content of C_func.x.array
    # No new array is created, the matrix->cell mapping is done here.
    c_values[local_idx, :, :] = np.vstack((Ci@T, np.zeros((padding_size, mat.shape[1]))))
C_func.x.scatter_forward()

In [ ]:
V_spline = create_spline_space(cells_to_dofs=padded_cells_to_dofs, mesh=disconnected_mesh, N_max=N_max, cell_type="hexahedron", dtype=dtype)
tabulate_A = make_bilinear_kernel(disconnected_mesh, a0, padded_dofs=N_max, local_dofs=local_dofs_size)
tabulate_b = make_linear_kernel(disconnected_mesh, f0, padded_dofs=N_max, local_dofs=local_dofs_size)

In [ ]:
formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[V_spline._cpp_object, 
                V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[C_func._cpp_object
                    ], # weights w_, holds C@T
              constants=[],
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.cell: [(0, tabulate_b.address, cells, np.array([0], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object], # test space, determines the size of b_
        integrals=integrals_rhs, #give the adress of tabulate_b
        coefficients=[C_func._cpp_object], # holds the evaluations of f at the correct points, as well as C@T
        constants=[], need_permutation_data=False, entity_maps=[], mesh=msh._cpp_object
    )
)

In [ ]:
x_vec, A = solve_problem(hs=hs, a=a_cond, rhs=l_cond, dummy_index=np.max(padded_cells_to_dofs),
                         V_spline = V_spline, iterative=True, return_A=True, dirichlet_indices=None)

u_dg = map_spline_to_legendre(hs, V, C_func, N_max, disconnected_mesh, padded_cells_to_dofs, x_vec)
u_dg.x.scatter_forward()


# Compute exact L2 error using FEniCSx standard UFL
error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) * dx_custom)
error_sq = dolfinx.fem.assemble_scalar(error_form)
exact_l2_error = np.sqrt(disconnected_mesh.comm.allreduce(error_sq, op=MPI.SUM))

print(f"Exact L2 Error (via DG projection): {exact_l2_error:.3e}")
rel_err = exact_l2_error/f_sq_integral
print(f"Relative error = {rel_err:.3e}")
A.destroy()
# print(f"dofs = {A.getSize()}")
V_error = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
v = ufl.TestFunction(V_error)

hQ = ufl.CellDiameter(disconnected_mesh)
volume_form = dolfinx.fem.form(1.0*v*dx_custom)
cell_volumes = dolfinx.fem.assemble_vector(volume_form).array

# Define the local L2 error form: integral of (f - u_dg)^2 per cell
# Note: We multiply by the test function 'v' to pick out each cell's contribution
local_error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f-u_dg) * v * dx_custom)

err_cells = dorfler_marking(hierarchical_space=hs, theta=0.3, local_error_form=local_error_form)